# Ghost 3D Forge — modo 1 clique
Ative **GPU T4** no Colab e execute somente a célula abaixo. Ela monta o Google Drive, reaproveita cache de downloads, instala o SF3D, pede seu token do Hugging Face, solicita a imagem e baixa o GLB no final.

**Importante:** o SF3D oficial gera a partir de uma imagem por vez. Use a melhor vista frontal nesta etapa.


In [ ]:
#@title ▶️ INICIAR GHOST 3D FORGE
import os, sys, subprocess, shutil, glob, pathlib, torch
from getpass import getpass

print('=== Ghost 3D Forge ===')
if not torch.cuda.is_available():
    raise RuntimeError('GPU não está ativa. No Colab escolha Ambiente de execução > Alterar tipo de ambiente > T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive, files
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/Ghost3DForge/cache'
REPO_CACHE = '/content/drive/MyDrive/Ghost3DForge/stable-fast-3d'
os.makedirs(CACHE, exist_ok=True)
os.environ['HF_HOME'] = CACHE + '/huggingface'
os.environ['PIP_CACHE_DIR'] = CACHE + '/pip'

print('Preparando dependências...')
subprocess.run(['apt-get','-qq','update'], check=True)
subprocess.run(['apt-get','-qq','install','-y','git','build-essential','libgl1','libglib2.0-0'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','setuptools==69.5.1','wheel','huggingface_hub'], check=True)

if not os.path.isdir(REPO_CACHE + '/.git'):
    print('Baixando SF3D pela primeira vez para o seu Drive...')
    shutil.rmtree(REPO_CACHE, ignore_errors=True)
    subprocess.run(['git','clone','--depth','1','https://github.com/Stability-AI/stable-fast-3d.git', REPO_CACHE], check=True)
else:
    print('SF3D encontrado no Google Drive. Reaproveitando arquivos.')

os.chdir(REPO_CACHE)
marker = CACHE + '/sf3d_requirements_v1.ok'
if not os.path.exists(marker):
    print('Instalando dependências do SF3D. A primeira vez ainda pode demorar.')
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
    pathlib.Path(marker).write_text('ok')
else:
    print('Cache detectado. Verificando dependências rapidamente...')
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)

from huggingface_hub import login
token = getpass('Cole seu token READ do Hugging Face: ')
login(token=token, add_to_git_credential=False)

print('Agora escolha a imagem.')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Nenhuma imagem enviada.')
input_name = next(iter(uploaded.keys()))
input_path = '/content/' + input_name
with open(input_path,'wb') as f:
    f.write(uploaded[input_name])

out_dir = '/content/ghost3d_output'
shutil.rmtree(out_dir, ignore_errors=True)
os.makedirs(out_dir, exist_ok=True)
print('Gerando GLB...')
cmd = [sys.executable,'run.py',input_path,'--output-dir',out_dir,'--texture-resolution','1024','--remesh_option','triangle']
subprocess.run(cmd, check=True)
glbs = glob.glob(out_dir + '/**/*.glb', recursive=True)
if not glbs:
    raise FileNotFoundError('O SF3D terminou sem gerar GLB.')
final_glb = glbs[0]
saved = '/content/drive/MyDrive/Ghost3DForge/resultados/' + pathlib.Path(final_glb).name
os.makedirs(os.path.dirname(saved), exist_ok=True)
shutil.copy2(final_glb, saved)
print('Pronto! Uma cópia também foi salva no Google Drive:', saved)
files.download(final_glb)


## Como usar depois
Em uma sessão nova: conecte uma **T4** e aperte somente **▶️ INICIAR GHOST 3D FORGE**. O repositório, cache do Hugging Face e cache do pip ficam em `Meu Drive/Ghost3DForge`, reduzindo downloads repetidos. O Colab ainda pode precisar reinstalar alguns pacotes porque a máquina virtual é descartável.
